# PacCafe Database Data Check

Notebook ini digunakan untuk mengecek data yang ada di masing-masing database pada project `mentoring_2`:

- Source database: `paccafe`
- Staging database: `staging`
- Warehouse database: `warehouse`
- Log database: `pipeline_log`

Notebook ini memakai command-line `psql` dari Python `subprocess`, sehingga tidak membutuhkan package tambahan seperti `psycopg`, `sqlalchemy`, atau `pandas`. Notebook dijalankan dari environment `uv` project.

Scope saat ini sudah end-to-end: PostgreSQL source tables dan copied spreadsheet data `store_branch_paccofee.csv` masuk ke staging lalu warehouse. Expected count untuk `store_branch` dan `dim_store_branch` adalah `3`.


## 1. Pre-check

Pastikan dependency notebook dan service Docker sudah siap. Jalankan dari root `mentoring_2`:

```bash
uv sync --group notebook
cd data_pipeline_paccafe
docker compose up -d
cd ..
```

Jika ingin refresh data sebelum inspeksi, jalankan pipeline dulu:

```bash
uv run paccafe-pipeline
```

Lalu buka notebook ini dengan:

```bash
uv run jupyter notebook notebooks/db_data_check.ipynb
```


In [12]:
import csv
import io
import os
import shutil
import subprocess
from IPython.display import Markdown, display

psql_path = shutil.which("psql")
if not psql_path:
    raise RuntimeError("psql is not available on PATH. Install PostgreSQL client first.")

print(f"psql found: {psql_path}")

psql found: /opt/homebrew/bin/psql


## 2. Database Configuration

Default value mengikuti `.env` yang dipakai Docker Compose.

In [13]:
DB_CONFIGS = {
    "source": {
        "database": os.getenv("SRC_POSTGRES_DB", "paccafe"),
        "host": os.getenv("SRC_POSTGRES_HOST", "localhost"),
        "port": os.getenv("SRC_POSTGRES_PORT", "5433"),
        "user": os.getenv("SRC_POSTGRES_USER", "postgres"),
        "password": os.getenv("SRC_POSTGRES_PASSWORD", "postgres"),
    },
    "staging": {
        "database": os.getenv("STG_POSTGRES_DB", "staging"),
        "host": os.getenv("STG_POSTGRES_HOST", "localhost"),
        "port": os.getenv("STG_POSTGRES_PORT", "5434"),
        "user": os.getenv("STG_POSTGRES_USER", "postgres"),
        "password": os.getenv("STG_POSTGRES_PASSWORD", "postgres"),
    },
    "warehouse": {
        "database": os.getenv("WH_POSTGRES_DB", "warehouse"),
        "host": os.getenv("WH_POSTGRES_HOST", "localhost"),
        "port": os.getenv("WH_POSTGRES_PORT", "5435"),
        "user": os.getenv("WH_POSTGRES_USER", "postgres"),
        "password": os.getenv("WH_POSTGRES_PASSWORD", "postgres"),
    },
    "log": {
        "database": os.getenv("LOG_POSTGRES_DB", "pipeline_log"),
        "host": os.getenv("LOG_POSTGRES_HOST", "localhost"),
        "port": os.getenv("LOG_POSTGRES_PORT", "5436"),
        "user": os.getenv("LOG_POSTGRES_USER", "postgres"),
        "password": os.getenv("LOG_POSTGRES_PASSWORD", "postgres"),
    },
}

def db_url(db_key):
    cfg = DB_CONFIGS[db_key]
    return f"postgresql://{cfg['user']}:{cfg['password']}@{cfg['host']}:{cfg['port']}/{cfg['database']}"

for name, cfg in DB_CONFIGS.items():
    print(f"{name}: {cfg['host']}:{cfg['port']}/{cfg['database']}")

source: localhost:5433/paccafe
staging: localhost:5434/staging
warehouse: localhost:5435/warehouse
log: localhost:5436/pipeline_log


## 3. Helper Functions

In [14]:
def run_psql(db_key, sql, *, csv_output=True):
    if csv_output:
        sql = f"COPY ({sql}) TO STDOUT WITH CSV HEADER"
    command = [
        "psql",
        db_url(db_key),
        "-X",
        "-q",
        "-v",
        "ON_ERROR_STOP=1",
        "-c",
        sql,
    ]
    result = subprocess.run(command, text=True, capture_output=True, check=False)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return result.stdout


def fetch_rows(db_key, sql):
    output = run_psql(db_key, sql, csv_output=True)
    return list(csv.DictReader(io.StringIO(output)))


def quote_identifier(identifier):
    if not identifier.replace("_", "").isalnum():
        raise ValueError(f"Unsafe identifier: {identifier!r}")
    return '"' + identifier.replace('"', '""') + '"'


def markdown_table(rows, max_rows=20):
    if not rows:
        return "_No rows returned._"
    rows = rows[:max_rows]
    headers = list(rows[0].keys())
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        values = [str(row.get(header, "")).replace("\n", " ") for header in headers]
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)


def show_rows(rows, title=None, max_rows=20):
    if title:
        display(Markdown(f"### {title}"))
    display(Markdown(markdown_table(rows, max_rows=max_rows)))


def list_tables(db_key):
    return fetch_rows(
        db_key,
        """
        SELECT table_schema, table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
          AND table_type = 'BASE TABLE'
        ORDER BY table_name
        """,
    )


def row_counts(db_key):
    tables = list_tables(db_key)
    results = []
    for table in tables:
        table_name = table["table_name"]
        count_rows = fetch_rows(db_key, f"SELECT count(*) AS row_count FROM public.{quote_identifier(table_name)}")
        results.append({"table_name": table_name, "row_count": count_rows[0]["row_count"]})
    return results


def sample_table(db_key, table_name, limit=5):
    return fetch_rows(db_key, f"SELECT * FROM public.{quote_identifier(table_name)} LIMIT {int(limit)}")

## 4. Connection Check

In [15]:
connection_rows = []
for db_key in DB_CONFIGS:
    rows = fetch_rows(
        db_key,
        """
        SELECT
            current_database() AS database,
            current_user AS user_name,
            inet_server_port() AS server_port
        """,
    )
    connection_rows.append({"db_key": db_key, **rows[0]})

show_rows(connection_rows, "Connection Result")

### Connection Result

| db_key | database | user_name | server_port |
| --- | --- | --- | --- |
| source | paccafe | postgres | 5432 |
| staging | staging | postgres | 5432 |
| warehouse | warehouse | postgres | 5432 |
| log | pipeline_log | postgres | 5432 |

## 5. List Tables per Database

In [16]:
for db_key in DB_CONFIGS:
    show_rows(list_tables(db_key), f"Tables in {db_key}")

### Tables in source

| table_schema | table_name |
| --- | --- |
| public | customers |
| public | employees |
| public | inventory_tracking |
| public | order_details |
| public | orders |
| public | products |

### Tables in staging

| table_schema | table_name |
| --- | --- |
| public | customers |
| public | employees |
| public | inventory_tracking |
| public | order_details |
| public | orders |
| public | products |
| public | store_branch |

### Tables in warehouse

| table_schema | table_name |
| --- | --- |
| public | dim_customers |
| public | dim_date |
| public | dim_employees |
| public | dim_products |
| public | dim_store_branch |
| public | fct_inventory |
| public | fct_order |

### Tables in log

| table_schema | table_name |
| --- | --- |
| public | etl_log |

## 6. Row Count per Database

In [17]:
all_counts = {}
for db_key in DB_CONFIGS:
    counts = row_counts(db_key)
    all_counts[db_key] = counts
    show_rows(counts, f"Row Counts in {db_key}")

### Row Counts in source

| table_name | row_count |
| --- | --- |
| customers | 204 |
| employees | 103 |
| inventory_tracking | 162 |
| order_details | 3022 |
| orders | 1010 |
| products | 54 |

### Row Counts in staging

| table_name | row_count |
| --- | --- |
| customers | 204 |
| employees | 103 |
| inventory_tracking | 162 |
| order_details | 3022 |
| orders | 1010 |
| products | 54 |
| store_branch | 3 |

### Row Counts in warehouse

| table_name | row_count |
| --- | --- |
| dim_customers | 204 |
| dim_date | 29220 |
| dim_employees | 103 |
| dim_products | 54 |
| dim_store_branch | 3 |
| fct_inventory | 162 |
| fct_order | 1010 |

### Row Counts in log

| table_name | row_count |
| --- | --- |
| etl_log | 25 |

## 7. Source Database Sample Data

In [18]:
for table_name in ["customers", "employees", "orders", "order_details", "products", "inventory_tracking"]:
    show_rows(sample_table("source", table_name, limit=5), f"source.public.{table_name}", max_rows=5)

### source.public.customers

| customer_id | first_name | last_name | email | phone | loyalty_points | created_at |
| --- | --- | --- | --- | --- | --- | --- |
| 492 | Paul | Hawkins | bmorris@example.net | 9958687561434 | 708 | 2022-06-13 03:01:06 |
| 493 | Jessica | Hendricks | sandramalone@example.com | 3378843199040 | 241 | 2020-09-08 21:31:15 |
| 494 | Christian | Shepherd | lreed@example.net | 8814427344096 | 239 | 2023-03-27 18:02:37 |
| 495 | Robert | Perry | ninalucero@example.com | 6819873900353 | 956 | 2024-05-20 19:48:10 |
| 496 | Courtney | Miller | coxmartin@example.net | 7185028607446 | 677 | 2020-01-05 06:32:44 |

### source.public.employees

| employee_id | first_name | last_name | hire_date | role | email | created_at |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | John | Gordon | 2020-07-19 | Cashier | john04@example.net | 2020-07-19 07:48:20 |
| 2 | Gregory | Curry | 2023-05-15 | Waitress | rachaelfreeman@example.org | 2023-05-15 02:28:15 |
| 3 | Chris | Allen | 2022-11-27 | Waitress | petercox@example.org | 2022-11-27 14:52:27 |
| 4 | Ryan | Mitchell | 2021-07-06 | Cashier | smithsteven@example.net | 2021-07-06 06:00:29 |
| 5 | Tyler | Wiggins | 2021-10-21 | Manager | vgriffin@example.com | 2021-10-21 21:23:43 |

### source.public.orders

...

| order_detail_id | order_id | product_id | quantity | unit_price | subtotal | created_at |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | 1001 | 80 | 1 | 15.00 | 15.00 | 2023-01-03 10:42:46 |
| 2 | 1001 | 65 | 8 | 18.00 | 144.00 | 2023-01-03 10:42:46 |
| 3 | 1002 | 57 | 4 | 43.00 | 172.00 | 2023-01-03 15:25:52 |
| 4 | 1002 | 58 | 2 | 9.00 | 18.00 | 2023-01-03 15:25:52 |
| 5 | 1002 | 69 | 6 | 41.00 | 246.00 | 2023-01-03 15:25:52 |

### source.public.products

| product_id | product_name | category | unit_price | cost_price | in_stock | created_at |
| --- | --- | --- | --- | --- | --- | --- |
| 53 | Pastry situation | Pastry | $47.00 | $43.00 | t | 2024-09-21 00:00:00 |
| 54 | Pastry sure | Pastry | $47.00 | $43.00 | t | 2022-10-20 00:00:00 |
| 55 | Pastry statement | Pastry | $17.00 | $14.00 | t | 2024-01-17 00:00:00 |
| 56 | Salad on | Salad | $21.00 | $20.00 | t | 2020-08-27 00:00:00 |
| 57 | Salad really | Salad | $43.00 | $40.00 | t | 2020-05-08 00:00:00 |

### source.public.inventory_tracking

| tracking_id | product_id | quantity_change | change_date | reason | created_at |
| --- | --- | --- | --- | --- | --- |
| 1 | 53 | 3 | 2024-11-01 00:00:00 | Restock | 2024-11-01 00:00:00 |
| 2 | 54 | 10 | 2024-08-05 00:00:00 | Restock | 2024-08-05 00:00:00 |
| 3 | 55 | 6 | 2023-11-12 00:00:00 | Damaged | 2023-11-12 00:00:00 |
| 4 | 56 | 8 | 2023-05-29 00:00:00 | Damaged | 2023-05-29 00:00:00 |
| 5 | 57 | 6 | 2023-09-22 00:00:00 | Restock | 2023-09-22 00:00:00 |

## 8. Staging Database Sample Data

In [19]:
for table_name in ["customers", "employees", "orders", "order_details", "products", "inventory_tracking", "store_branch"]:
    show_rows(sample_table("staging", table_name, limit=5), f"staging.public.{table_name}", max_rows=5)

### staging.public.customers

| customer_id | first_name | last_name | email | phone | loyalty_points | created_at |
| --- | --- | --- | --- | --- | --- | --- |
| 492 | Paul | Hawkins | bmorris@example.net | 9958687561434 | 708 | 2022-06-13 03:01:06 |
| 493 | Jessica | Hendricks | sandramalone@example.com | 3378843199040 | 241 | 2020-09-08 21:31:15 |
| 494 | Christian | Shepherd | lreed@example.net | 8814427344096 | 239 | 2023-03-27 18:02:37 |
| 495 | Robert | Perry | ninalucero@example.com | 6819873900353 | 956 | 2024-05-20 19:48:10 |
| 496 | Courtney | Miller | coxmartin@example.net | 7185028607446 | 677 | 2020-01-05 06:32:44 |

### staging.public.employees

| employee_id | first_name | last_name | hire_date | role | email | created_at |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | John | Gordon | 2020-07-19 | Cashier | john04@example.net | 2020-07-19 07:48:20 |
| 2 | Gregory | Curry | 2023-05-15 | Waitress | rachaelfreeman@example.org | 2023-05-15 02:28:15 |
| 3 | Chris | Allen | 2022-11-27 | Waitress | petercox@example.org | 2022-11-27 14:52:27 |
| 4 | Ryan | Mitchell | 2021-07-06 | Cashier | smithsteven@example.net | 2021-07-06 06:00:29 |
| 5 | Tyler | Wiggins | 2021-10-21 | Manager | vgriffin@example.com | 2021-10-21 21:23:43 |

### staging.public.orders

...

| product_id | product_name | category | unit_price | cost_price | in_stock | store_branch | created_at |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 53 | Pastry situation | Pastry | $47.00 | $43.00 | t |  | 2024-09-21 00:00:00 |
| 54 | Pastry sure | Pastry | $47.00 | $43.00 | t |  | 2022-10-20 00:00:00 |
| 55 | Pastry statement | Pastry | $17.00 | $14.00 | t |  | 2024-01-17 00:00:00 |
| 56 | Salad on | Salad | $21.00 | $20.00 | t |  | 2020-08-27 00:00:00 |
| 57 | Salad really | Salad | $43.00 | $40.00 | t |  | 2020-05-08 00:00:00 |

### staging.public.inventory_tracking

| tracking_id | product_id | quantity_change | change_date | reason | created_at |
| --- | --- | --- | --- | --- | --- |
| 1 | 53 | 3 | 2024-11-01 00:00:00 | Restock | 2024-11-01 00:00:00 |
| 2 | 54 | 10 | 2024-08-05 00:00:00 | Restock | 2024-08-05 00:00:00 |
| 3 | 55 | 6 | 2023-11-12 00:00:00 | Damaged | 2023-11-12 00:00:00 |
| 4 | 56 | 8 | 2023-05-29 00:00:00 | Damaged | 2023-05-29 00:00:00 |
| 5 | 57 | 6 | 2023-09-22 00:00:00 | Restock | 2023-09-22 00:00:00 |

### staging.public.store_branch

| store_id | store_name | created_at |
| --- | --- | --- |
| 1 | Dapur Kenangan | 2025-01-31 20:43:18 |
| 2 | Laci Coffee | 2025-01-31 20:43:18 |
| 3 | Setara Coffee | 2025-01-31 20:43:18 |

## 9. Warehouse Database Sample Data

In [20]:
for table_name in ["dim_customers", "dim_employees", "dim_products", "dim_store_branch", "fct_order", "fct_inventory"]:
    show_rows(sample_table("warehouse", table_name, limit=5), f"warehouse.public.{table_name}", max_rows=5)

### warehouse.public.dim_customers

| sk_customer_id | nk_customer_id | first_name | last_name | email | phone | loyalty_points | created_at |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 24ccdb36-c580-43a6-bf1b-d64f37aeac24 | 492 | Paul | Hawkins | bmorris@example.net | 9958687561434 | 708 | 2022-06-13 03:01:06 |
| 22d897b4-657a-4464-9310-d6d7748a411f | 493 | Jessica | Hendricks | sandramalone@example.com | 3378843199040 | 241 | 2020-09-08 21:31:15 |
| d2330766-902b-45ba-9e17-71cdb54158c1 | 494 | Christian | Shepherd | lreed@example.net | 8814427344096 | 239 | 2023-03-27 18:02:37 |
| d3762a28-d116-482b-a096-19220cec6b67 | 495 | Robert | Perry | ninalucero@example.com | 6819873900353 | 956 | 2024-05-20 19:48:10 |
| 25c7f7b3-aa2b-4f4e-ad02-9fd2874fa063 | 496 | Courtney | Miller | coxmartin@example.net | 7185028607446 | 677 | 2020-01-05 06:32:44 |

### warehouse.public.dim_employees

| sk_employee_id | nk_employee_id | first_name | last_name | hire_date | role | email | created_at |
| --- | --- | --- | --- | --- | --- | --- | --- |
| bf855767-d1d7-4666-a7ad-6ad4675ccfaa | 1 | John | Gordon | 20200719 | Cashier | john04@example.net | 2020-07-19 07:48:20 |
| b922bc73-8485-4391-b37f-cff8f93ec80b | 2 | Gregory | Curry | 20230515 | Waitress | rachaelfreeman@example.org | 2023-05-15 02:28:15 |
| 0e2e417b-1c6a-4f00-b920-f6a021e82936 | 3 | Chris | Allen | 20221127 | Waitress | petercox@example.org | 2022-11-27 14:52:27 |
| e5d97d46-54c0-49ad-b031-59c6f9b765b7 | 4 | Ryan | Mitchell | 20210706 | Cashier | smithsteven@example.net | 2021-07-06 06:00:29 |
| 9096ad56-5bb6-4e6d-ad06-d52a38a565ea | 5 | Tyler | Wiggins | 20211021 | Manager | vgriffin@example.com | 2021-10-21 21:23:43 |

### warehouse.public.dim_products

...

| sk_store_id | nk_store_id | store_name | created_at |
| --- | --- | --- | --- |
| 47c4f4da-b86b-46c8-b417-cb10e7bcc214 | 1 | Dapur Kenangan | 2025-01-31 20:43:18 |
| 9eeacdd2-4527-4737-aae7-d2c3bc742c85 | 2 | Laci Coffee | 2025-01-31 20:43:18 |
| 91f419aa-ea73-4015-823a-348d9b0a046a | 3 | Setara Coffee | 2025-01-31 20:43:18 |

### warehouse.public.fct_order

| sk_order_id | nk_order_id | sk_employee_id | sk_customer_id | sk_product_id | order_date | total_amount | quantity | unit_price | subtotal | payment_method | order_status | created_at |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| f78b4cc9-7484-4131-82d9-3a3e6d9b2de8 | 1484 | dfb9b888-eb79-47a7-b636-25f0d64460b4 |  | 4d1fd08b-1739-4bb7-8b5b-c16ae4644545 | 20231204 | 442.00 | 21 | 21.0476190476190476 | 442.00 | Cash | Completed | 2023-12-04 02:28:08 |
| bade2941-70ce-4a42-a66e-52a09b2f8b04 | 1798 | 7288d503-97b6-436e-9a48-fb8edf305905 | 7c5a9d48-4e32-4526-b249-2123d83ec1e2 | 235fa776-5c58-41cd-9fb5-e601d7dba486 | 20240803 | 344.00 | 14 | 24.5714285714285714 | 344.00 | Cash | Completed | 2024-08-03 15:33:09 |
| 538e3499-61a0-43d7-a05d-541ca70791b0 | 1489 | c72f82b6-1d12-4667-9208-db9f866a7dfd | 9a79de2f-edb2-4e95-8b2d-25e962816540 | 25f21589-4f8b-4373-b99d-6e0502f59311 | 20231213 | 465.00 | 22 | 21.1363636363636364 | 465.00 | Credit Card | Completed | 2023-12-13 08:26:34 |
| b16bb065-8298-4bcf-8982-02391f8fef53 | 1269 | c72f82b6-1d12-4667-9208-db9f866a7dfd |  | c1a638a5-af6d-4df6-a111-16b9119c7e52 | 20230714 | 292.00 | 12 | 24.3333333333333333 | 292.00 | Debit Card | Completed | 2023-07-14 02:20:42 |
| cc3ae3ae-0588-4bcb-9d2e-3423054d503c | 1989 | 5897bccc-f52f-4b15-8218-8e6f2584a058 | 95f3f049-da2b-40c5-9a2d-ee7576c397c2 | 106b078f-8634-48de-acce-7d9909c94208 | 20241221 | 35.00 | 1 | 35.0000000000000000 | 35.00 | Cash | Completed | 2024-12-21 14:49:27 |

### warehouse.public.fct_inventory

| sk_tracking_id | nk_tracking_id | sk_product_id | quantity_change | change_date | reason | created_at |
| --- | --- | --- | --- | --- | --- | --- |
| b631b175-e540-4b39-afcf-ea61a046a083 | 1 | 2a71b95a-dca1-4422-8c95-15d144f1f9ca | 3 | 20241101 | Restock | 2024-11-01 00:00:00 |
| c69d67fe-0267-4760-9d8e-0be624096d8b | 2 | 6171b11a-c407-4f25-83c0-ceacc2ae4b15 | 10 | 20240805 | Restock | 2024-08-05 00:00:00 |
| df6028de-5c77-4e6d-9ccc-18e411474b5b | 3 | c1a638a5-af6d-4df6-a111-16b9119c7e52 | 6 | 20231112 | Damaged | 2023-11-12 00:00:00 |
| 5e2934c1-998e-4d13-b21d-9f6dae87d920 | 4 | 25f21589-4f8b-4373-b99d-6e0502f59311 | 8 | 20230529 | Damaged | 2023-05-29 00:00:00 |
| 0318347a-b9b8-4455-bf23-3729f2672e8b | 5 | 01da306d-805c-40b0-ad5d-b8893267c265 | 6 | 20230922 | Restock | 2023-09-22 00:00:00 |

## 10. Log Database Sample Data

In [21]:
show_rows(
    fetch_rows(
        "log",
        """
        SELECT step, component, status, table_name, etl_date, error_msg
        FROM public.etl_log
        ORDER BY etl_date DESC
        LIMIT 20
        """,
    ),
    "Latest ETL Logs",
    max_rows=20,
)

### Latest ETL Logs

| step | component | status | table_name | etl_date | error_msg |
| --- | --- | --- | --- | --- | --- |
| warehouse | load_warehouse_dimensions_and_facts | SUCCESS |  | 2026-05-28 13:41:58.28252 | [{"table_name": "dim_customers", "row_count": "204"}, {"table_name": "dim_employees", "row_count": "103"}, {"table_name": "dim_products", "row_count": "54"}, {"table_name": "dim_store_branch", "row_count": "3"}, {"table_name": "fct_inventory", "row_count": "162"}, {"table_name": "fct_order", "row_count": "1010"}] |
| warehouse | load_warehouse_dimensions_and_facts | STARTED |  | 2026-05-28 13:41:58.108853 |  |
| warehouse | sync_staging_tables_to_warehouse | SUCCESS |  | 2026-05-28 13:41:58.079264 |  |
| warehouse | sync_staging_tables_to_warehouse:store_branch | SUCCESS | store_branch | 2026-05-28 13:41:58.052175 | row_count=3 |
| warehouse | sync_staging_tables_to_warehouse:inventory_tracking | SUCCESS | inventory_tracking | 2026-05-28 13:41:57.883094 | row_count=162 |
| warehouse | sync_staging_tables_to_warehouse:products | SUCCESS | products | 2026-05-28 13:41:57.700507 | row_count=54 |
| warehouse | sync_staging_tables_to_warehouse:order_details | SUCCESS | order_details | 2026-05-28 13:41:57.526948 | row_count=3022 |
| warehouse | sync_staging_tables_to_warehouse:orders | SUCCESS | orders | 2026-05-28 13:41:57.343565 | row_count=1010 |
| warehouse | sync_staging_tables_to_warehouse:employees | SUCCESS | employees | 2026-05-28 13:41:57.15745 | row_count=103 |
| warehouse | sync_staging_tables_to_warehouse:customers | SUCCESS | customers | 2026-05-28 13:41:56.999515 | row_count=204 |
| warehouse | sync_staging_tables_to_warehouse | STARTED |  | 2026-05-28 13:41:56.798145 |  |
| staging | load_store_branch_spreadsheet_to_staging | SUCCESS | store_branch | 2026-05-28 13:41:56.768795 | row_count=3 |
| staging | load_store_branch_spreadsheet_to_staging | STARTED | store_branch | 2026-05-28 13:41:56.668177 |  |
| staging | load_inventory_tracking_to_staging | SUCCESS | inventory_tracking | 2026-05-28 13:41:56.638773 | row_count=162 |
| staging | load_inventory_tracking_to_staging | STARTED | inventory_tracking | 2026-05-28 13:41:56.470613 |  |
| staging | load_products_to_staging | SUCCESS | products | 2026-05-28 13:41:56.437354 | row_count=54 |
| staging | load_products_to_staging | STARTED | products | 2026-05-28 13:41:56.258802 |  |
| staging | load_order_details_to_staging | SUCCESS | order_details | 2026-05-28 13:41:56.230833 | row_count=3022 |
| staging | load_order_details_to_staging | STARTED | order_details | 2026-05-28 13:41:56.03308 |  |
| staging | load_orders_to_staging | SUCCESS | orders | 2026-05-28 13:41:56.002682 | row_count=1010 |

## 11. Warehouse Quality Checks

In [22]:
quality_checks = fetch_rows(
    "warehouse",
    """
    SELECT 'fct_order_bad_dates' AS check_name, count(*) AS issue_count
    FROM public.fct_order
    WHERE order_date NOT IN (SELECT date_id FROM public.dim_date)
    UNION ALL
    SELECT 'fct_inventory_bad_dates', count(*)
    FROM public.fct_inventory
    WHERE change_date NOT IN (SELECT date_id FROM public.dim_date)
    UNION ALL
    SELECT 'dim_customers_duplicate_nk', count(*)
    FROM (
        SELECT nk_customer_id
        FROM public.dim_customers
        GROUP BY nk_customer_id
        HAVING count(*) > 1
    ) duplicates
    UNION ALL
    SELECT 'dim_products_duplicate_nk', count(*)
    FROM (
        SELECT nk_product_id
        FROM public.dim_products
        GROUP BY nk_product_id
        HAVING count(*) > 1
    ) duplicates
    """,
)

show_rows(quality_checks, "Warehouse Quality Checks")

### Warehouse Quality Checks

| check_name | issue_count |
| --- | --- |
| fct_order_bad_dates | 0 |
| fct_inventory_bad_dates | 0 |
| dim_customers_duplicate_nk | 0 |
| dim_products_duplicate_nk | 0 |

## 12. Notes

- `store_branch` dan `dim_store_branch` expected bernilai `3` karena data spreadsheet copy `store_branch_paccofee.csv` sudah masuk pipeline.
- Notebook ini hanya untuk inspeksi data, bukan untuk menjalankan ETL.
- Untuk menjalankan ETL, gunakan salah satu command berikut dari root `mentoring_2`:

```bash
uv run paccafe-pipeline
```

atau:

```bash
uv run python -m paccafe_pipeline
```
